- Chạy local thì cài đặt sẵn Anaconda hoặc Miniconda để tạo môi trường
B1: Mở anaconda prompt lên tại thư mục project, đảm bảo ổ cứng chứa thư mu
B2: Chạy lệnh '''conda env create -f environment.yml'''
B3: Chạy lệnh '''conda activate aio_hutech'''
B4: Chạy lệnh '''!pip install kagglehub'''
B5:
- Chạy local mà có sẵn CONDA:
    + Thêm dòng <tensorflow=2.19.0> trong chỗ dependencies file environment.yml trước khi mở anaconda prompt
- Chạy local mà không có CUDA:
    Chạy lệnh '''pip install tensorflow==2.19.0'''
B6: Load model về

In [ ]:
!pip install kagglehub

In [ ]:
!pip install tensorflow==2.19.0

In [15]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import os
import cv2
import keras
import numpy as np
import albumentations as A
import random
import tf_keras as keras
import tensorflow_hub as hub
import kagglehub # Đánh comment khi chạy trên kaggle
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input
from tensorflow.keras.models import Model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2B0
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.image import save_img
from PIL import Image
from keras.layers import TFSMLayer
from keras import Sequential


In [3]:
#Thông số các biến
batch_size = 128
learning_rate = 0.005
image_shape = (32, 32)
# Tạo ánh xạ nhãn theo thứ tự mong muốn
label_map = {
    "nấm mỡ": 0,
    "bào ngư xám + trắng": 1,
    "Đùi gà Baby (cắt ngắn)": 2,
    "linh chi trắng": 3
}

In [7]:
#Đọc file ảnh từ thư mục
input_test = r"D:\Downloads\AI_HUTECH\aio-hutech\test"
output_test = r"D:\Downloads\AI_HUTECH\aio-hutech\submission.csv"
input_train_original = r"D:\Downloads\AI_HUTECH\aio-hutech\train"

input_train = r"D:\Downloads\AI_HUTECH\aio-hutech_custom\train"
output_train = r"D:\Downloads\AI_HUTECH\aio-hutech_custom\train_augment"
input_val = r"D:\Downloads\AI_HUTECH\aio-hutech_custom\val";
output_val = r"D:\Downloads\AI_HUTECH\aio-hutech_custom\val_augment"

if not os.path.exists(input_val):
    os.makedirs(input_val)
if not os.path.exists(output_train):
    os.makedirs(output_train)
if not os.path.exists(input_train):
    os.makedirs(input_train)
if not os.path.exists(output_val):
    os.makedirs(output_val)

In [19]:
# Chạy trên kaggle không cần chạy phần này
# Tải mô hình EfficientNetV2 từ KaggleHub
path = "https://www.kaggle.com/models/google/efficientnet-v2/TensorFlow2/imagenet1k-b0-classification/2"

print("Path to model files:", path)

Path to model files: https://www.kaggle.com/models/google/efficientnet-v2/TensorFlow2/imagenet1k-b0-classification/2


In [ ]:
# Xây dựng transfer learning model
def create_model(model_url, num_classes):
 
  feature_extractor_layer = hub.KerasLayer(model_url,
                                           trainable=False,
                                           name='feature_extraction_layer',
                                           input_shape=image_shape+(3,)) 

 
  model = keras.Sequential([
    feature_extractor_layer, 
    keras.layers.Dense(num_classes, activation='sigmoid', name='output_layer') 
      
  ])
  return model

# Function of copying images
def copy_image(input_folder, output_folder, first_number, last_number):
    for class_name in os.listdir(input_folder):
        class_path = os.path.join(input_folder, class_name)
        if not os.path.isdir(class_path):
            continue

        output_class_path = os.path.join(output_folder, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        for image_name in os.listdir(class_path)[first_number:last_number]:
            image_path = os.path.join(class_path, image_name)

            # Load image safely (unicode support)
            try:
                pil_image = Image.open(image_path).convert("RGB")
            except Exception as e:
                print(f"Cannot open image: {image_path}, error: {e}")
                continue

            image = np.array(pil_image)

            save_path_original = os.path.join(output_class_path, image_name)
            save_img(save_path_original, image)
            print(f"Saved: {save_path_original}")

# Function of augmentation data training and validation
def augment_image(input_folder, output_folder):
    transform = A.Compose([
        A.OneOf([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5)
        ], p=0.8),

        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.8),

        A.OneOf([
            A.GaussianBlur(blur_limit=3, p=0.5),
            A.resized_crop(height=128, width=124),
        ], p=0.5),

        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),

        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.5)
    ])

    # For each class
    for class_name in os.listdir(input_folder):
        class_path = os.path.join(input_folder, class_name)
        if not os.path.isdir(class_path):
            continue

        output_class_path = os.path.join(output_folder, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        for image_name in os.listdir(class_path):
            image_path = os.path.join(class_path, image_name)

            # Load image safely (unicode support)
            try:
                pil_image = Image.open(image_path).convert("RGB")
            except Exception as e:
                print(f"Cannot open image: {image_path}, error: {e}")
                continue

            image = np.array(pil_image)

            # Save original image
            save_path_original = os.path.join(output_class_path, f"orig_{image_name}")
            save_img(save_path_original, image)
            print(f"Saved: {save_path_original}")

            #  5 transforms cố định
            for i in range(5):
                augmented = transform(image=image)
                aug_image = augmented["image"]
                save_path = os.path.join(output_class_path, f"aug{i+1}_{image_name}")
                save_img(save_path, aug_image)
                print(f"Saved: {save_path}")

In [ ]:
# Lấy 250 ảnh cuối từ train_original vào train_custom
copy_image(input_train_original, input_train, 50, 300)
# Lấy 50 ảnh đầu từ train_original vào val_custom
copy_image(input_train_original, input_val, 0, 50)

# Augmentation data training and validation
augment_image(input_train, output_train)
augment_image(input_val, output_val)

In [ ]:
datagen = ImageDataGenerator()
# Load dữ liệu train
train_generator = datagen.flow_from_directory(
    output_train,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=True
)

# Load dữ liệu val
val_generator = datagen.flow_from_directory(
    output_val,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='sparse',
    shuffle=True
)

# Load tập test
test_generator = datagen.flow_from_directory(
    input_test,
    target_size=(128, 128),
    batch_size=1,
    class_mode=None,
    shuffle=False
)

# Cập nhật lại nhãn theo ánh xạ tùy chỉnh
# Lấy tên class từ đường dẫn file
train_generator.class_indices = label_map
train_generator.classes = np.array([
    label_map[os.path.dirname(file)] for file in train_generator.filenames
])

val_generator.class_indices = label_map
val_generator.classes = np.array([
    label_map[os.path.dirname(file)] for file in val_generator.filenames
])

# Kiểm tra xem nhãn đã đúng chưa
print(train_generator.class_indices)
print(val_generator.class_indices)

Found 6000 images belonging to 4 classes.
Found 1200 images belonging to 4 classes.
Found 0 images belonging to 0 classes.
{'nấm mỡ': 0, 'bào ngư xám + trắng': 1, 'Đùi gà Baby (cắt ngắn)': 2, 'linh chi trắng': 3}
{'nấm mỡ': 0, 'bào ngư xám + trắng': 1, 'Đùi gà Baby (cắt ngắn)': 2, 'linh chi trắng': 3}


In [20]:
model_url = path
input_tensor = Input(shape=(32, 32, 3))  # Định nghĩa input size 32x32
base_model = hub.KerasLayer(model_url, trainable=False)(input_tensor)

# 2. Thêm các tầng Fully Connected
x = GlobalAveragePooling2D()(base_model)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
out = Dense(1, activation='sigmoid')  # Bài toán nhị phân

model = Model(inputs=input_tensor, outputs=out)

# 3. Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# 5. Huấn luyện
model.fit(train_generator, validation_data=val_generator, epochs=10)

# 6. Lưu kiến trúc (KHÔNG lưu trọng số)
with open("model_architecture.json", "w") as json_file:
    json_file.write(model.to_json())

ValueError: Exception encountered when calling layer 'keras_layer_1' (type KerasLayer).

A KerasTensor is symbolic: it's a placeholder for a shape an a dtype. It doesn't have any actual numerical value. You cannot convert it to a NumPy array.

Call arguments received by layer 'keras_layer_1' (type KerasLayer):
  • inputs=<KerasTensor shape=(None, 32, 32, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1>
  • training=None

In [ ]:
# Compile model trước khi train
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
# Huấn luyện mô hình
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=5,
    validation_data=val_generator,
    validation_steps=len(val_generator)
)


In [ ]:
# Vẽ biểu đồ Loss và Accuracy
plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss over epochs')

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy over epochs')

plt.show()

In [ ]:
# Dự đoán trên tập validation
y_true = val_generator.classes  # Nhãn thật
y_pred = model.predict(val_generator)
y_pred = np.argmax(y_pred, axis=1)  # Chuyển sang dạng class index

# Vẽ confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=train_generator.class_indices, yticklabels=train_generator.class_indices)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()
